# Мерей: воспроизведение stage 2 — ae2e088 → 766d0a39

Notebook закрепляет **candidate 766d0a39fe8394be8a1cd280401af1adde6d5206**.
Предыдущий runtime Мерея — **ae2e0887a5179ed602b6e8bc7b7538f518791a31**.
Основное performance-сравнение — previous Мерей → candidate; optional ABBA
включается ниже. Этот файл подготовлен без выполнения и не содержит
измеренных результатов или заранее объявленного ускорения.

Kaggle: **GPU T4 x2**, Internet On, свежая сессия, свободные GPU, Run All.
Обучение использует только CUDA 0 через исходный harness. GPU subprocesses
запускаются последовательно. Предел candidate GPU peak — **5175 MiB**.

Зависимости настраиваются **один раз, до любых Transformers/TRL imports**.
После setup пакеты и metadata не меняются. Native metadata parity через реальный Soup setup-prefix выполняется
в отдельном CPU child и требует ноль native factory calls; тесты и обучение — в собственных свежих процессах.
Исходные измерения выполняют только `harness/run_task.py` и его subprocesses:
новые timers/monitors или готовые compiler caches не добавляются.

Native Soup запускается только в обязательной admission/quality процедуре.
Её сравнение требуется правилами и не заменяет previous-Мерей A/B.

GPU скрыта только в отдельном CPU diagnostic child. Обучение сохраняет исходное GPU-окружение и отдельно проверяется по полному peak памяти.


In [ ]:
import hashlib, importlib.metadata as metadata, json, os
import shutil, statistics, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/promaladoi/Decentrathon-6.0.git'
CANDIDATE = '766d0a39fe8394be8a1cd280401af1adde6d5206'
PREVIOUS_MEREI = 'ae2e0887a5179ed602b6e8bc7b7538f518791a31'
RUN_PREVIOUS_MEREI_AB = False  # True adds previous,new,new,previous diagnostic runs.
MEMORY_CEILING_MIB = 5175
WORKING = Path('/kaggle/working')
EVIDENCE = WORKING / ('merey-stage2-' + CANDIDATE[:12] + '-evidence')
EVIDENCE.mkdir(exist_ok=False)

gpu_names = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True)
print(gpu_names, flush=True)
assert 'T4' in gpu_names, 'Choose GPU T4 x2'
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass


In [ ]:
def installed(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

# The controller uses distribution metadata, not imports of these libraries.
# Finish every install/removal here before any test or training child starts.
assert installed('torch') == '2.11.0+cu128', 'Use the recorded Kaggle image'
assert installed('triton') == '3.6.0', 'Use Triton 3.6.0 from the recorded image'
requirements = {
    'soup-cli': ('0.77.0', 'soup-cli[train]==0.77.0'),
    'transformers': ('5.16.1', 'transformers==5.16.1'),
    'peft': ('0.20.0', 'peft==0.20.0'),
    'trl': ('0.29.1', 'trl==0.29.1'),
    'bitsandbytes': ('0.50.2', 'bitsandbytes==0.50.2'),
    'accelerate': ('1.14.0', 'accelerate==1.14.0'),
    'datasets': ('4.8.5', 'datasets==4.8.5'),
}
missing = [spec for name, (required, spec) in requirements.items()
           if installed(name) != required]
if installed('pytest') is None:
    missing.append('pytest')
if installed('torchao') is not None:
    # Remove the optional incompatible package before the fixed environment begins.
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
versions = {name: installed(name) for name in ['torch', 'triton', *requirements, 'pytest']}
assert all(versions[name] == required for name, (required, spec) in requirements.items())
assert versions['torch'] == '2.11.0+cu128' and versions['triton'] == '3.6.0'
print(json.dumps(versions, indent=2), flush=True)
# No later cell installs, removes, upgrades or edits packages.


In [ ]:
def clone_exact(label, commit):
    checkout = WORKING / ('merey-stage2-' + label + '-' + commit[:12])
    assert not checkout.exists(), 'Preserve previous results and use a fresh session'
    subprocess.run(['git', 'clone', '-q', '--no-checkout', REPO_URL, str(checkout)], check=True)
    subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', commit], check=True)
    actual = subprocess.check_output(
        ['git', '-C', str(checkout), 'rev-parse', 'HEAD'], text=True).strip()
    assert actual == commit
    return checkout / 'Мерей'

roots = {'candidate': clone_exact('candidate', CANDIDATE)}
if RUN_PREVIOUS_MEREI_AB:
    roots['previous'] = clone_exact('previous', PREVIOUS_MEREI)
    for directory in ['harness', 'task', 'tests']:
        a, b = roots['previous'] / directory, roots['candidate'] / directory
        files = sorted(p.relative_to(a) for p in a.rglob('*') if p.is_file())
        assert files == sorted(p.relative_to(b) for p in b.rglob('*') if p.is_file())
        assert all((a / p).read_bytes() == (b / p).read_bytes() for p in files), directory

sys.path.insert(0, str(roots['candidate'] / 'harness'))
from common import ENV
assert ENV['CUDA_VISIBLE_DEVICES'] == '0', 'Use the original one-GPU harness environment'

def run_repo(label, arguments, log_name, **overrides):
    print('RUN_BEGIN', label, arguments, flush=True)
    log = EVIDENCE / log_name
    with log.open('w', encoding='utf-8') as stream:
        result = subprocess.run([sys.executable, *arguments], cwd=roots[label],
                                env=dict(ENV, **overrides), stdout=stream,
                                stderr=subprocess.STDOUT)
    output = log.read_text(encoding='utf-8')
    print(output, flush=True)
    result.check_returncode()
    return output

def solution_hash(root, python_only=False):
    digest = hashlib.sha256()
    for path in sorted((root / 'solution').rglob('*')):
        if (path.is_file() and '__pycache__' not in path.parts
                and (not python_only or path.suffix == '.py')):
            digest.update(path.relative_to(root / 'solution').as_posix().encode())
            digest.update(path.read_bytes())
    return digest.hexdigest()

measured_solution_git_tree = subprocess.check_output(
    ['git', '-C', str(roots['candidate'].parent), 'rev-parse',
     CANDIDATE + ':Мерей/solution'], text=True).strip()
assert len(measured_solution_git_tree) == 40

provenance = {
    'candidate_runtime_commit': CANDIDATE,
    'previous_merei_runtime_commit': PREVIOUS_MEREI,
    'versions': versions,
    'candidate_solution_sha256_before_tests': solution_hash(roots['candidate']),
    'candidate_python_sha256': solution_hash(roots['candidate'], python_only=True),
    'measured_solution_git_tree': measured_solution_git_tree,
    'memory_ceiling_mib': MEMORY_CEILING_MIB,
}
if RUN_PREVIOUS_MEREI_AB:
    provenance['previous_solution_sha256'] = solution_hash(roots['previous'])
(EVIDENCE / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
print(json.dumps(provenance, indent=2), flush=True)


## Реальный native setup-prefix и полные тесты

Отдельный CPU child сначала импортирует настоящие Transformers и Soup,
проверяет отсутствие раннего TRL import и устанавливает обёртку исходного
`SFTTrainerWrapper.setup`. Затем вызывается реальный setup; sentinel на
первом чтении config останавливает его после исходного TRL import, до seed
и model construction.

Profiler должен подтвердить **один вызов обёртки, один вызов native setup,
один фактический snapshot return в TRL и ноль вызовов исходной
packages_distributions от любых callers во время этого prefix**.
Возвращённый словарь должен быть именно тем mapping, который получил TRL.
После восстановления stdlib-функции весь mapping сравнивается со свежим
native scan; проверяются независимость списков и результаты исходных helpers.
Этот CPU diagnostic не входит в training measurements и не получает GPU.

Затем выполняются весь `solution/` suite и все **79 Soup regressions**
с включённым решением. Для закреплённого solution suite ожидаются
**352 passed + 1 skipped**; пропускаемый native test отдельно выполняется
выше как **1 native diagnostic**. Counts не зашиты в assertions: фактические
passed/skipped записываются в логи. Подготовка данных использует исходный
task script.


In [ ]:
parity_output = run_repo(
    'candidate', ['solution/test_package_metadata_snapshot.py', 'NativeInstalledTests'],
    'native-metadata-parity.log',
    MEREY_NATIVE_METADATA_SNAPSHOT_PARITY='1', CUDA_VISIBLE_DEVICES='')
prefix = 'native metadata snapshot parity: '
records = [json.loads(line.split(prefix, 1)[1])
           for line in parity_output.splitlines() if prefix in line]
assert len(records) == 1, 'The explicit installed-native parity diagnostic must run'
parity = records[0]
assert parity['native_mapping_equal'] and parity['independent_dict_and_lists']
assert parity['function_restored'] and not parity['cuda_initialized']
assert parity['cpu_probe_gpu_hidden']
assert parity['setup_calls'] == {'wrapper': 1, 'native': 1}
assert parity['actual_snapshot_returns_to_trl'] == 1
assert parity['native_scan_callers_during_setup'] == []
assert parity['stopped_before_seed_and_model'] and parity['setup_idempotent']
(EVIDENCE / 'native-metadata-parity.json').write_text(
    json.dumps(parity, indent=2), encoding='utf-8')


In [ ]:
run_repo('candidate', ['-m', 'pytest', 'solution', '-q', '-p', 'no:cacheprovider'],
         'solution-tests.log')
run_repo('candidate', ['-m', 'pytest', 'tests', '-q', '-p', 'no:cacheprovider'],
         'soup-regressions.log', SOUP_APPLY_SOLUTION='1')
for label in roots:
    run_repo(label, ['task/prepare_data.py'], label + '-prepare-data.log')
assert solution_hash(roots['candidate']) == provenance['candidate_solution_sha256_before_tests']


## Обязательный admission/quality run

Неизменённый official harness выполняет base → solution → solution → base,
eval и все десять rule checks, включая Soup regressions. Native Soup здесь
используется для обязательных quality, settings, outputs и admission checks.
Результаты этой процедуры не объявляются ускорением относительно предыдущего
Мерея.

Пакеты, исходники и настройки между запусками не меняются. Каждый training
subprocess получает оригинальные CUDA 0, allocator и пустые compiler caches
от harness. Candidate дополнительно ограничен **5175 MiB**.


In [ ]:
ADMISSION = 'runs_reproduce_stage2_766_admission'
run_repo('candidate', ['harness/run_task.py', '--repeats', '2', '--out', ADMISSION],
         'official-training.log')
run_repo('candidate', ['harness/check_rules.py', '--result', ADMISSION + '/result.json'],
         'official-checks.log')
result_path = roots['candidate'] / ADMISSION / 'result.json'
checks_path = roots['candidate'] / ADMISSION / 'checks.json'
result = json.loads(result_path.read_text())
checks = json.loads(checks_path.read_text())
assert len(checks['checks']) == 10 and all(item['ok'] for item in checks['checks'])
assert checks['result_sha256'] == hashlib.sha256(result_path.read_bytes()).hexdigest()
assert checks['solution_sha256'] == solution_hash(roots['candidate'])
assert checks['solution_sha256'] == provenance['candidate_solution_sha256_before_tests']
candidate = result['summary']['solution']
assert candidate['gpu_peak_mib'] <= MEMORY_CEILING_MIB, 'Stage2 memory ceiling exceeded'
shutil.copy2(result_path, EVIDENCE / 'final-result.json')
shutil.copy2(checks_path, EVIDENCE / 'final-checks.json')
print('ADMISSION_COMPLETE', json.dumps({
    'candidate_runtime_commit': CANDIDATE, 'candidate_summary': candidate,
    'checks_passed': 10, 'memory_ceiling_mib': MEMORY_CEILING_MIB,
}, indent=2), flush=True)


## Optional: previous Мерей ae2e088 против candidate 766d0a39

Включите `RUN_PREVIOUS_MEREI_AB=True` в первой code cell до Run All.
Этот этап выполняет **previous → candidate → candidate → previous** без
native Soup. Сравниваются исходные `ae2e0887…` и `766d0a39…` в одной
сессии и фиксированном environment. Это performance diagnostic, а не замена
official admission.

Whole-command time, train-loop time и steps 3–64 выводятся отдельно.
Сокращение startup/save time не объявляется ускорением steady-state.
`--only solution` не используется для полного rule check.


In [ ]:
if RUN_PREVIOUS_MEREI_AB:
    ab_runs = {'previous': [], 'candidate': []}
    order = [('previous', 'old_1'), ('candidate', 'new_1'),
             ('candidate', 'new_2'), ('previous', 'old_2')]
    for label, name in order:
        folder = 'runs_reproduce_stage2_ab_' + name
        run_repo(label, ['harness/run_task.py', '--only', 'solution', '--out', folder],
                 'ab-' + name + '.log')
        path = roots[label] / folder / 'result.json'
        report = json.loads(path.read_text())
        assert report['runs']['solution'] and all(r['exit_code'] == 0 for r in report['runs']['solution'])
        ab_runs[label].extend(report['runs']['solution'])
        shutil.copy2(path, EVIDENCE / ('ab-' + name + '-result.json'))

    def summarize(rows):
        return {
            'wall_runs_s': [r['wall_s'] for r in rows],
            'wall_median_s': statistics.median(r['wall_s'] for r in rows),
            'gpu_peak_mib': max(r['gpu_peak_mib'] for r in rows),
            'train_loop_median_s': statistics.median(r['metrics']['train_loop_s'] for r in rows),
            'timed_steps_3plus_median_s': statistics.median(r['metrics']['timed_loop_s'] for r in rows),
        }
    summary = {label: summarize(rows) for label, rows in ab_runs.items()}
    assert summary['candidate']['gpu_peak_mib'] <= MEMORY_CEILING_MIB
    assert summary['candidate']['gpu_peak_mib'] <= summary['previous']['gpu_peak_mib']
    comparison = {
        'kind': 'previous_merei_ab_diagnostic_not_admission',
        'previous_commit': PREVIOUS_MEREI, 'candidate_commit': CANDIDATE,
        'order': [name for label, name in order], 'summary': summary,
        'whole_time_reduction_pct': 100 * (
            1 - summary['candidate']['wall_median_s'] / summary['previous']['wall_median_s']),
    }
    (EVIDENCE / 'previous-merei-ab.json').write_text(
        json.dumps(comparison, indent=2), encoding='utf-8')
    print(json.dumps(comparison, indent=2), flush=True)
else:
    print('Optional previous-Merei A/B disabled; mandatory admission completed above.')


## Evidence и exact source hashes

`provenance.json`, real-setup native metadata parity, test logs, `final-result.json` и
`final-checks.json` лежат в напечатанной папке evidence. Optional A/B добавляет
четыре исходных result files и `previous-merei-ab.json`. Сохраните Kaggle
version с output и скачайте evidence. Не публикуйте Kaggle Secrets.

Notebook воспроизводит точные bytes **runtime
766d0a39fe8394be8a1cd280401af1adde6d5206**. Новый reproduction template
публикуется в `Мерей/templates/stage2_optimization_validation.ipynb`,
**вне solution/**. Дерево `Мерей/solution` остаётся точной копией измеренного
winner, включая прежние stage1 notes и notebook; source SHA не перепривязывается
после изменения документов внутри solution.

`provenance.json` записывает полный solution SHA256, production Python SHA256
и Git tree измеренного `Мерей/solution`. Отдельный публикационный
`source-proof.json` должен проверять точное равенство:
**published Solution Git tree == measured766 Solution Git tree**.
Изменение любых solution bytes нарушает эту связь и требует новой проверки
исходников и измерений. Этот подготовленный notebook сам по себе не является
успешным Kaggle run или admission proof.


In [ ]:
print('EVIDENCE_DIRECTORY', str(EVIDENCE), flush=True)
print('MEREY_STAGE2_766_REPRODUCTION_COMPLETE', flush=True)
